In [4]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2"
os.environ["WANDB_DISABLED"] = "true"

from accelerate import notebook_launcher

def train_distributed():
    import torch
    from datasets import load_from_disk
    from transformers import set_seed

    # Your modules
    from tokenizer import ScImmuneTokenizer
    from config import ScImmuneConfig
    from model import ScImmuneModel
    from collator import ScImmuneDataCollator
    from trainer import ScImmunePretrainingTrainer, ScImmuneTrainingArguments

    set_seed(42)

    # ----- SETTINGS YOU CAN TUNE -----
    tokenized_data_path = "scimmune-model/tokenized_data"  # HF dataset saved via save_to_disk
    vocab_file = "vocab_with_metadata.json"
    base_model_dir = "scImmune_metadata_model/"           # has config.json + model.safetensors
    output_dir = "runs/scimmune-ctpt"
    num_metadata_tokens = 6
    MAX_LEN = 2000     # pick 1200–1536 unless your sequences demand more
    # ---------------------------------

    # 1) Tokenizer
    tokenizer = ScImmuneTokenizer(vocab_file=vocab_file)

    # 2) Load dataset
    ds = load_from_disk(tokenized_data_path)

    # 2a) Ensure zeros for metadata tokens in "values" (or "expressions") and move <cls> to front
    #     Only run these maps if you haven't already persisted them.
    cls_token_id = tokenizer.convert_tokens_to_ids("<cls>")

    def ensure_prefix_and_zeros(example):
        # prepend zeros for metadata tokens to the "values" field
        vals = example.get("values", None)
        if vals is None:
            vals = example["expressions"]
        if isinstance(vals, list):
            vals = torch.tensor(vals, dtype=torch.float32)
        zeros = torch.zeros(num_metadata_tokens, dtype=vals.dtype)
        vals = torch.cat((zeros, vals))
        example["expressions"] = vals  # normalize to "expressions" name

        # move <cls> to index 0 in "genes"
        genes = example["genes"]
        if isinstance(genes, list):
            genes = torch.tensor(genes, dtype=torch.long)
        # if cls exists (it should), move it to front
        cls_positions = (genes == cls_token_id).nonzero(as_tuple=True)[0]
        if len(cls_positions) > 0 and cls_positions[0].item() != 0:
            idx = cls_positions[0].item()
            genes = torch.cat([genes[idx:idx+1], genes[:idx], genes[idx+1:]])
        example["genes"] = genes
        return example

    ds = ds.map(ensure_prefix_and_zeros)

    # 2b) Keep only the columns we need in torch format
    ds = ds.with_format(type="torch", columns=["genes","expressions"])
    splits = ds.train_test_split(test_size=0.02, shuffle=True, seed=42)
    train_dataset, eval_dataset = splits["train"], splits["test"]

    # 3) Collator — continuous values (no binning)
    collator = ScImmuneDataCollator(
        do_padding=True,
        pad_token_id=tokenizer.pad_token_id,   # <- do not hardcode
        pad_value=-2,                          # padded expr sentinel
        do_mlm=True,
        do_binning=False,                      # continuous pipeline
        mlm_probability=0.15,
        mask_value=-1,                         # masked expr sentinel
        max_length=MAX_LEN,
        sampling=True,
        keep_first_n_tokens=1 + num_metadata_tokens,  # <cls> + metadata protected
        data_style="both",                     # perceptual + generative
    )

    # 4) Config & Model — align with collator
    cfg = ScImmuneConfig.from_pretrained(
        base_model_dir,
        input_emb_style="continuous",          # must match do_binning=False
        pad_value=-2,
        mask_value=-1,
        use_generative_training=True,          # because data_style="both"
    )
    cfg.vocab_size   = len(tokenizer)
    cfg.pad_token_id = tokenizer.pad_token_id
    # (optional) make sure model can handle MAX_LEN if you use positional limits internally
    if hasattr(cfg, "max_seq_len"):
        cfg.max_seq_len = max(getattr(cfg, "max_seq_len", MAX_LEN), MAX_LEN)

    model = ScImmuneModel.from_pretrained(base_model_dir, config=cfg)
    model.resize_token_embeddings(len(tokenizer))

    # 5) Quick sanity check on a collated batch (rank 0 only)
    #    This helps catch key/shape mismatches early.
    # try:
    #     from torch.utils.data import DataLoader
    #     b = next(iter(DataLoader(train_dataset, batch_size=4, collate_fn=collator)))
    #     # Keys expected in BOTH mode:
    #     assert {"pcpt_gene","pcpt_expr","masked_expr",
    #             "gen_gene","gen_expr_target",
    #             "pcpt_key_padding_mask","gen_key_padding_mask"} <= set(b.keys())
    #     # Prefix (unmasked) check
    #     K = 1 + num_metadata_tokens
    #     assert torch.equal(b["masked_expr"][:, :K], b["pcpt_expr"][:, :K])
    #     # Pad sentinels check
    #     assert (b["pcpt_expr"][b["pcpt_key_padding_mask"]] == -2).all()
    # except Exception as e:
    #     print("[Sanity check] Collator batch failed:", e)
    #     raise

    # 6) Training args & Trainer
    training_args = ScImmuneTrainingArguments(
        output_dir=output_dir,
        per_device_train_batch_size=8,     # per GPU; adjust if you OOM
        per_device_eval_batch_size=8,
        gradient_accumulation_steps=4,     # effective batch = 8 * GPUs * 4
        learning_rate=1e-4,
        weight_decay=0.01,
        max_steps=50_000,                  # or set num_train_epochs
        lr_scheduler_type="cosine",
        warmup_ratio=0.10,
        logging_steps=50,
        save_steps=1000,
        eval_steps=1000,
        evaluation_strategy="steps",
        save_total_limit=3,
        fp16=True,
        dataloader_num_workers=4,
        dataloader_pin_memory=True,
        seed=42,
        remove_unused_columns=False,
        # scGPT-specific flags the collator/trainer use
        mlm_probability=0.15,
        max_length=MAX_LEN,
        MVC=False,
        # ddp_find_unused_parameters=False,  # uncomment if all branches used
    )

    trainer = ScImmunePretrainingTrainer(
        model=model,
        args=training_args,
        data_collator=collator,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
    )

    # 7) Train & save
    trainer.train()
    trainer.save_model(os.path.join(output_dir, "final"))

# Launch 3 processes (3 GPUs) from the notebook
notebook_launcher(train_distributed, args=(), num_processes=3, mixed_precision="fp16")

Launching training on 3 CUDAs.


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
W0809 18:44:06.601289 1827089 torch/multiprocessing/spawn.py:169] Terminating process 1861737 via signal SIGTERM
W0809 18:44:06.602640 1827089 torch/multiprocessing/spawn.py:169] Terminating process 1861738 via signal SIGTERM
W0809 18:44:36.633335 1827089 torch/multiprocessing/spawn.py:180] Unable to shutdown process 1861737 via SIGTERM , forcefully exiting via SIGKILL
E0809 18:44:36.859495 1827089 t

ChildFailedError: 
============================================================
train_distributed FAILED
------------------------------------------------------------
Failures:
  <NO_OTHER_FAILURES>
------------------------------------------------------------
Root Cause (first observed failure):
[0]:
  time      : 2025-08-09_18:44:06
  host      : sc-srv12.sdsc.edu
  rank      : 2 (local_rank: 2)
  exitcode  : 1 (pid: 1861739)
  error_file: /tmp/torchelastic_ezf_p200/none_ma741zfa/attempt_0/2/error.json
  traceback : Traceback (most recent call last):
    File "/home/s5srinivasan/immune-foundational-model/.venv/lib64/python3.9/site-packages/torch/distributed/elastic/multiprocessing/errors/__init__.py", line 357, in wrapper
      return f(*args, **kwargs)
    File "/tmp/ipykernel_1827089/738934644.py", line 157, in train_distributed
      trainer.train()
    File "/home/s5srinivasan/immune-foundational-model/.venv/lib64/python3.9/site-packages/transformers/trainer.py", line 2238, in trai